In [277]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn import set_config


# 1

In [278]:
geo_data_0 = pd.read_csv("../datasets/geo_data_0.csv")
geo_data_1 = pd.read_csv("../datasets/geo_data_1.csv")
geo_data_2 = pd.read_csv("../datasets/geo_data_2.csv")


# 2

In [279]:
def features_target_split(data_df:pd.DataFrame,non_training_columns:list[str],target_column:str) -> tuple[pd.DataFrame,pd.Series]:
    features = data_df.drop(non_training_columns + [target_column],axis=1)
    target = data_df.loc[:,target_column]
    return features, target

In [280]:
def model_pipe(num_columns:list[str]):
    scaler = StandardScaler()
    ct = ColumnTransformer(
        transformers=[("num_preprocess", scaler, num_columns)],
        remainder="passthrough"
    )

    pipe = Pipeline([
        ("column_transform", ct),
        ("model", LinearRegression())
    ])
    return pipe

In [281]:
def model_training(features_train,features_valid,target_train,target_valid,pipe:Pipeline) -> tuple[float, float,Pipeline]:
    pipe.fit(features_train,target_train)
    predicted_valid = pd.Series(pipe.predict(features_valid))
    mean_predicted = predicted_valid.mean()
    root_mean_square_error = mean_squared_error(target_valid,predicted_valid) ** (1/2)
    return mean_predicted, root_mean_square_error, pipe



In [282]:
def region_workflow(data_df:pd.DataFrame,non_training_columns,feature_columns,target_column,test_size:float,random_state:int) -> tuple[pd.Series,pd.Series,pd.DataFrame,float,float,Pipeline]:

    features, target = features_target_split(data_df, non_training_columns,target_column)
    features_train, features_valid, target_train, target_valid = train_test_split(features, target, test_size=test_size,random_state=random_state)

    pipe = model_pipe(feature_columns)
    pipe.fit(features_train,target_train)

    predicted_valid = pd.Series(pipe.predict(features_valid),index=target_valid.index)
    validation_results = pd.DataFrame({"actual_product":target_valid,"predicted_product":predicted_valid}).merge(data_df[non_training_columns],how="left",left_index=True,right_index=True)

    mean_predicted = predicted_valid.mean()
    root_mean_square_error = mean_squared_error(target_valid,predicted_valid) ** (1/2)

    return target_valid, predicted_valid, validation_results, mean_predicted,root_mean_square_error,pipe

In [283]:
split_state = 12345
train_size = 0.75
valid_size = 0.25
non_training_columns = ["id"]
feature_columns = ["f0","f1","f2"]
target_column = "product"


In [284]:
target_valid_0, predicted_valid_0, validation_results_0, mean_predicted_0, rmse_0, pipe_0 = region_workflow(geo_data_0,non_training_columns,feature_columns,target_column,valid_size,split_state)
print(f"Volume médio previsto de reservas: {mean_predicted_0:.2f}")
print(f"REQM do modelo: {rmse_0:.2f}")


Volume médio previsto de reservas: 92.59
REQM do modelo: 37.58


In [285]:
target_valid_1, predicted_valid_1, validation_results_1, mean_predicted_1, rmse_1, pipe_1 = region_workflow(geo_data_1,non_training_columns,feature_columns,target_column,valid_size,split_state)
print(f"Volume médio previsto de reservas: {mean_predicted_1:.2f}")
print(f"REQM do modelo: {rmse_1:.2f}")


Volume médio previsto de reservas: 68.73
REQM do modelo: 0.89


In [286]:
target_valid_2, predicted_valid_2, validation_results_2, mean_predicted_2, rmse_2, pipe_2 = region_workflow(geo_data_2,non_training_columns,feature_columns,target_column,valid_size,split_state)
print(f"Volume médio previsto de reservas: {mean_predicted_2:.2f}")
print(f"REQM do modelo: {rmse_2:.2f}")


Volume médio previsto de reservas: 94.97
REQM do modelo: 40.03


# Análise

- Região 0 tem volume médio previsto de reservas de 92.59 milhares de barris de petróleo com REQM 37.58, o que indica que as predições são incertas.

- Região 1 tem volume médio previsto de reservas de 68.73 milhares de barris de petróleo com REQM 0.89, o que indica que as predições tem alto nível de confiabilidade.

- Região 2 tem volume médio previsto de reservas de 94.97 milhares de barris de petróleo com REQM 40.03, o que indica que as predições são incertas.

Nenhuma das três regiões pode ser escolhida como a melhor região para desenvolvimento usando somente essas informações. Mas podemos afirmar que, aplicando a metodologia de treinamento desses modelos, apenas o model oda região 1 parece ser confiável para futuros poços (assumindo que as características região não mudem e que os dados usados para treinamento são representativos da região).

# 3.

In [287]:
# Número de poços de Petróleo para desenvolvimento
oil_well_num = 200

# Orçamento de 1 milhões de dólares ($1.000.000) para o desenvolvimento
budget = 1e8

# Receita de uma unidade de produto (que equivalem a 1.000 barris de petróleo).
product_unit_rev = 4500

# Dado o investimento de 100 milhões para 200 poços de petróleo, cada um precisa
# produzir, em média, uma quantidade de unidades equivalente a pelo menos 500 mil
# dólares para evitar prejuízos (isso é aproximadamente 111,1 unidades).
min_mean_product = 111.11

# Risco de perdas máximo. Manter apenas as regiões com risco inferior.
max_loss_risk = 2.5


In [288]:
print(geo_data_0["product"].mean())
print(geo_data_1["product"].mean())
print(geo_data_2["product"].mean())

92.50000000000001
68.82500000000002
95.00000000000004


# 4,

In [289]:
best_wells_0 = geo_data_0.sort_values(by="product",ascending=False).head(200)
best_wells_1 = geo_data_1.sort_values(by="product",ascending=False).head(200)
best_wells_2 = geo_data_2.sort_values(by="product",ascending=False).head(200)